# A little playground to compare RSVD to scikit-learn

In [2]:
import sys
# Add the desired directory to sys.path
sys.path.append('build/')

import randSVD
from sklearn.utils.extmath import randomized_svd

A test matrix

In [5]:
import numpy as np

class TestMatrix:
    def __init__(self, rows, cols, sing_val_decay):
        self.rows = rows
        self.cols = cols
        self.sing_val_decay = sing_val_decay
        
        # Calculate rank
        self.rank = min(rows, cols)
        
        # Generate random U and V matrices
        self.U = np.random.rand(rows, self.rank)
        self.V = np.random.rand(cols, self.rank)
        
        # Orthogonalize U and V using QR decomposition
        self.U, _ = np.linalg.qr(self.U)
        self.V, _ = np.linalg.qr(self.V)
        
        # Set the singular values based on the decay option
        if self.sing_val_decay == "fast":
            self.sing_vals = np.power(0.95, np.arange(self.rank))
        else:
            self.sing_vals = 1/np.log(2 + np.arange(self.rank))

        self.A = (self.U @ np.diag(self.sing_vals) @ self.V.T).astype(np.float64)
        
    def matrixU(self):
        return self.U
    
    def singularValues(self):
        return self.sing_vals
    
    def matrixV(self):
        return self.V
    
    def matrixA(self):
        return self.A

In [9]:
def compute_errors(test_matrix,U,sing_vals,V):
    errors = {}

    rank = sing_vals.size

    errors["rec_error"] = np.linalg.norm(test_matrix.matrixA() - U@np.diag(sing_vals)@V.T, 'fro')
    errors["sing_val_error"] = np.linalg.norm(test_matrix.singularValues()[:rank] - sing_vals)

    errors["l_sing_vect_error"] = max(
        np.minimum(np.linalg.norm(test_matrix.matrixU()[:,:rank]-U,axis=0),
                   np.linalg.norm(test_matrix.matrixU()[:,:rank]+U,axis=0))
                   )    
    errors["r_sing_vect_error"] = max(
        np.minimum(np.linalg.norm(test_matrix.matrixV()[:,:rank]-V,axis=0),
                   np.linalg.norm(test_matrix.matrixV()[:,:rank]+V,axis=0))
                   )
    return errors

Testing

In [11]:
rank = 5  # to compare with rbki
n_iter = 10
tol = 1e-10
seed = 7050

test_matrix = TestMatrix(1000,1000,"slow")

rsi = randSVD.RSI(tol,n_iter,seed)
rsi.compute(test_matrix.matrixA(),rank,2*rank)
print(compute_errors(test_matrix, rsi.matrixU(), rsi.singularValues(), rsi.matrixV()))

U, s, Vh = randomized_svd(test_matrix.matrixA(),
                        n_components=rank,
                        n_oversamples=rank,
                        n_iter=n_iter,
                        random_state=seed)

print(compute_errors(test_matrix,U,s,Vh.T))

{'rec_error': 5.643736576703697, 'sing_val_error': 1.8751852954617226e-08, 'l_sing_vect_error': 0.00037445995554680184, 'r_sing_vect_error': 0.00027065761180315175}
{'rec_error': 5.6437365808921065, 'sing_val_error': 6.223889746750173e-08, 'l_sing_vect_error': 0.0006919051776174135, 'r_sing_vect_error': 0.0005056731921330978}
